# 08c — Community Summaries: Leiden + GraphRAG-Style LLM Summaries (B3)

**Phase 8c** — runs after `08b_citation_linker_entailment.ipynb`.

## What this notebook does

1. **Pre-flight** — Neo4j connectivity, CHUNK–KEYWORD edge count, igraph availability, existing COMMUNITY node count.
2. **Graph loading** — loads the CHUNK–KEYWORD bipartite graph from Neo4j into igraph (`load_bipartite_graph`).
3. **Leiden community detection** — projects to CHUNK-only graph, runs Leiden, reports size distribution.
4. **Community size histogram** — shows how many communities fall in each size bucket.
5. **Smoke summary run** — calls `build_community_summaries(limit=10_000)` with a capped edge load for a fast smoke run (≈ 20–50 communities).
6. **Community node inspection** — queries `(:COMMUNITY)` nodes, shows IDs, sizes, summaries, and embedding norms.
7. **IN_COMMUNITY edge check** — confirms `(:CHUNK)-[:IN_COMMUNITY]->(:COMMUNITY)` edges written.
8. **Community vector search probe** — embeds a sample query and searches `community_summary_embedding_index`.
9. **Artefact write** — saves `notebooks/_artifacts/08c_community_summaries/community_summaries.json`.

## Algorithm (plan §0.6 B3, §8)

```
CHUNK–KEYWORD bipartite graph (Neo4j MENTION edges)
  → igraph bipartite_projection (CHUNK side)
  → Leiden community detection (resolution=1.0, min_size=5)
  → per community: sample ≤10 chunk texts
  → deepseek-chat: ~200-token modern-Chinese summary
  → text-embedding-v4: embed summary
  → MERGE (:COMMUNITY {id, summary, embedding, size, modularity})
  → MERGE (:CHUNK)-[:IN_COMMUNITY]->(:COMMUNITY)
```

## Purpose in the search stack

Community summaries enable **global-query routing** (plan §9): for synthesis/interpretive queries
(intent = `interpretive-scholarly`), the search pipeline embeds the query, queries
`community_summary_embedding_index`, and retrieves community summaries as high-level context
before drilling down to individual CHUNKs — the same mechanism used by GraphRAG (Microsoft).

**Next**: `09_search_subgraph.ipynb` → `09b_verifier_full_normalization.ipynb` for the
full 2025-grade search stack wiring.


In [1]:
import json
import logging
import math
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "apps").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("neo4j.notifications").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "08c_community_summaries"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"artifact  : {ARTIFACT_DIR}")

REPO_ROOT : /Users/mohasani/Ancient
artifact  : /Users/mohasani/Ancient/notebooks/_artifacts/08c_community_summaries


In [2]:
from apps.backend.graph.neo4j_client import get_driver
from apps.backend.llm.silra import get_silra_client
from apps.backend.pipeline.community_summarize import (
    build_community_summaries,
    load_bipartite_graph,
    run_leiden,
    CommunityReport,
    _LEIDEN_RESOLUTION,
    _MIN_COMMUNITY_SIZE,
)
import os

driver = get_driver()
silra_client = get_silra_client()

LLM_MODEL   = os.getenv("LLM_MODEL", "deepseek-chat")
EMBED_MODEL = os.getenv("EMBED_LLM_MODEL", "text-embedding-v4")

print("Neo4j driver    : ready")
print(f"LLM model       : {LLM_MODEL}")
print(f"Embed model     : {EMBED_MODEL}")
print(f"Leiden resolution: {_LEIDEN_RESOLUTION}")
print(f"Min community   : {_MIN_COMMUNITY_SIZE} chunks")

Neo4j driver    : ready
LLM model       : deepseek-chat
Embed model     : text-embedding-v4
Leiden resolution: 1.0
Min community   : 5 chunks


## 1. Pre-flight checks


In [3]:
preflight: dict = {}

with driver.session() as s:
    r = s.run("RETURN 1 AS ok").single()
    preflight["neo4j_ok"] = bool(r and r["ok"] == 1)

    r2 = s.run(
        "MATCH ()-[m:MENTION]->() RETURN count(m) AS n"
    ).single()
    preflight["mention_edges"] = r2["n"]

    r3 = s.run(
        "MATCH (c:CHUNK) WHERE c.embedding IS NOT NULL "
        "RETURN count(c) AS n"
    ).single()
    preflight["chunks_with_embedding"] = r3["n"]

    r4 = s.run("MATCH (comm:COMMUNITY) RETURN count(comm) AS n").single()
    preflight["community_nodes_existing"] = r4["n"]

    r5 = s.run(
        "SHOW INDEXES YIELD name, type, state "
        "WHERE name = 'community_summary_embedding_index'"
    ).data()
    preflight["community_index_state"] = r5[0]["state"] if r5 else "MISSING"

# igraph availability
try:
    import igraph as ig
    preflight["igraph_version"] = ig.__version__
except ImportError:
    preflight["igraph_version"] = None

print(json.dumps(preflight, indent=2))

assert preflight["neo4j_ok"], "Neo4j not reachable"
assert preflight["mention_edges"] > 0, "No MENTION edges — run keyword extraction first"
assert preflight["igraph_version"] is not None, (
    "igraph not installed. Run: uv add igraph"
)

print("\n✅ Pre-flight passed")

{
  "neo4j_ok": true,
  "mention_edges": 270245,
  "chunks_with_embedding": 30159,
  "community_nodes_existing": 0,
  "community_index_state": "ONLINE",
  "igraph_version": "1.0.0"
}

✅ Pre-flight passed


## 2. Graph Loading

Load a manageable slice of the CHUNK–KEYWORD bipartite graph for the smoke run.  
The full graph has 270 K MENTION edges; `limit=10_000` gives a fast smoke.


In [4]:
SMOKE_LIMIT = 10_000  # edges to load for smoke run

print(f"Loading bipartite graph (limit={SMOKE_LIMIT:,} edges)...")
chunk_ids, edge_list, n_chunks, n_keywords = load_bipartite_graph(
    driver, limit=SMOKE_LIMIT
)

print(f"  Chunks   : {n_chunks:,}")
print(f"  Keywords : {n_keywords:,}")
print(f"  Edges    : {len(edge_list):,}")
print(f"  Density  : {len(edge_list) / max(n_chunks * n_keywords, 1):.6f}")

# Sample a few chunk IDs
print(f"\nSample chunk IDs (first 5):")
for cid in chunk_ids[:5]:
    print(f"  {cid}")

Loading bipartite graph (limit=10,000 edges)...
  Chunks   : 1,432
  Keywords : 4,750
  Edges    : 10,000
  Density  : 0.001470

Sample chunk IDs (first 5):
  刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00002::chunk_0000
  刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00003::chunk_0000
  刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00004::chunk_0000
  刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00005::chunk_0000
  刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00006::chunk_0000


## 3. Leiden Community Detection


In [5]:
print(f"Running Leiden (resolution={_LEIDEN_RESOLUTION}, min_size={_MIN_COMMUNITY_SIZE})...")

communities, modularity = run_leiden(
    chunk_ids, edge_list, n_chunks, n_keywords,
    resolution=_LEIDEN_RESOLUTION,
    min_size=_MIN_COMMUNITY_SIZE,
)

print(f"\nLeiden results:")
print(f"  Communities detected (size≥{_MIN_COMMUNITY_SIZE}): {len(communities)}")
print(f"  Modularity                                   : {modularity:.4f}")

if communities:
    sizes = [len(c) for c in communities]
    print(f"  Community sizes: min={min(sizes)} max={max(sizes)} mean={sum(sizes)/len(sizes):.1f}")
    print(f"  Total chunks covered: {sum(sizes):,} (with overlaps)")

2026-05-29 11:34:59,705 INFO     apps.backend.pipeline.community_summarize: Leiden: 5 total communities (≥5 members), modularity=0.4220


Running Leiden (resolution=1.0, min_size=5)...

Leiden results:
  Communities detected (size≥5): 5
  Modularity                                   : 0.4220
  Community sizes: min=6 max=522 mean=268.4
  Total chunks covered: 1,342 (with overlaps)


## 4. Community Size Histogram


In [6]:
if not communities:
    print("No communities — check MENTION edge density.")
else:
    sizes = sorted(len(c) for c in communities)
    buckets = [(5, 10), (11, 25), (26, 50), (51, 100), (101, 250), (251, 10_000)]

    print(f"Community size distribution ({len(communities)} communities):\n")
    print(f"  {'Range':>12}  {'Count':>6}  Bar")
    print("  " + "-" * 50)
    for lo, hi in buckets:
        count = sum(1 for s in sizes if lo <= s <= hi)
        bar   = "█" * min(count, 40)
        label = f"{lo}–{hi}" if hi < 10_000 else f"{lo}+"
        print(f"  {label:>12}  {count:>6}  {bar}")

    print()
    print(f"  Top 5 largest communities:")
    top5 = sorted(communities, key=len, reverse=True)[:5]
    for i, comm in enumerate(top5):
        print(f"    [{i+1}] {len(comm)} chunks — sample: {comm[0][:50]}")

Community size distribution (5 communities):

         Range   Count  Bar
  --------------------------------------------------
          5–10       1  █
         11–25       0  
         26–50       0  
        51–100       0  
       101–250       1  █
          251+       3  ███

  Top 5 largest communities:
    [1] 522 chunks — sample: 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00058::chunk_
    [2] 298 chunks — sample: 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00008::chunk_
    [3] 286 chunks — sample: 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00002::chunk_
    [4] 230 chunks — sample: 刘俊文_敦煌吐鲁番唐代法制文书考释_1989__7d163c80df::p00004::chunk_
    [5] 6 chunks — sample: 刘后滨_唐代告身的抄寫舆给付_唐研究第十四卷专号_天聖令及唐宋制度与社会研究__27d6d760eb


## 5. Smoke Summary Run

`limit=10_000` edges → ≈ 20–50 communities → deepseek-chat summaries + embeddings → write to Neo4j.


In [7]:
with driver.session() as s:
    n_comm_before = s.run("MATCH (c:COMMUNITY) RETURN count(c) AS n").single()["n"]
    n_member_before = s.run("MATCH ()-[r:IN_COMMUNITY]->() RETURN count(r) AS n").single()["n"]

print(f"COMMUNITY nodes before : {n_comm_before}")
print(f"IN_COMMUNITY edges before: {n_member_before}")
print(f"\nRunning build_community_summaries(limit={SMOKE_LIMIT:,})...\n")

report = build_community_summaries(
    driver,
    limit=SMOKE_LIMIT,
    resolution=_LEIDEN_RESOLUTION,
    min_size=_MIN_COMMUNITY_SIZE,
    llm_model=LLM_MODEL,
    embed_model=EMBED_MODEL,
)

with driver.session() as s:
    n_comm_after   = s.run("MATCH (c:COMMUNITY) RETURN count(c) AS n").single()["n"]
    n_member_after = s.run("MATCH ()-[r:IN_COMMUNITY]->() RETURN count(r) AS n").single()["n"]

print()
print("─" * 60)
print(json.dumps(report.to_dict(), indent=2, ensure_ascii=False))
print("─" * 60)
print(f"COMMUNITY nodes  : {n_comm_before} → {n_comm_after} (+{n_comm_after - n_comm_before})")
print(f"IN_COMMUNITY edges: {n_member_before} → {n_member_after} (+{n_member_after - n_member_before})")

2026-05-29 11:34:59,731 INFO     apps.backend.pipeline.community_summarize: Loading CHUNK-KEYWORD bipartite graph (limit=10000)…


2026-05-29 11:34:59,861 INFO     apps.backend.pipeline.community_summarize: Loaded 1432 chunks, 4750 keywords, 10000 edges


2026-05-29 11:34:59,861 INFO     apps.backend.pipeline.community_summarize: Running Leiden (resolution=1.00, min_size=5)…


2026-05-29 11:34:59,898 INFO     apps.backend.pipeline.community_summarize: Leiden: 5 total communities (≥5 members), modularity=0.4220


2026-05-29 11:34:59,899 INFO     apps.backend.pipeline.community_summarize: Detected 5 communities above min_size=5


COMMUNITY nodes before : 0
IN_COMMUNITY edges before: 0

Running build_community_summaries(limit=10,000)...



2026-05-29 11:35:33,045 INFO     apps.backend.pipeline.community_summarize: Embedding 5 community summaries…


2026-05-29 11:35:35,337 INFO     apps.backend.pipeline.community_summarize: Written 5 communities, 1342 IN_COMMUNITY edges



────────────────────────────────────────────────────────────
{
  "chunks_loaded": 1432,
  "edges_loaded": 10000,
  "communities_detected": 5,
  "communities_written": 5,
  "memberships_written": 1342,
  "duration_seconds": 35.61,
  "errors": []
}
────────────────────────────────────────────────────────────
COMMUNITY nodes  : 0 → 5 (+5)
IN_COMMUNITY edges: 0 → 1342 (+1342)


## 6. Community Node Inspection


In [8]:
with driver.session() as s:
    comm_rows = s.run(
        "MATCH (c:COMMUNITY) "
        "RETURN c.id AS id, c.size AS size, c.modularity AS mod, "
        "  c.summary AS summary, c.embedding AS emb "
        "ORDER BY c.size DESC LIMIT 10"
    ).data()

print(f"COMMUNITY nodes (top {len(comm_rows)} by size):\n")

community_sample: list[dict] = []

for row in comm_rows:
    emb = row.get("emb")
    emb_info = f"dims={len(emb)} norm={math.sqrt(sum(x*x for x in emb)):.4f}" if emb else "None"
    summary_preview = (row["summary"] or "")[:120].replace("\n", " ")
    print(
        f"  id={row['id'][:20]}…  size={row['size']:4d}  mod={row['mod']:.4f}\n"
        f"  emb: {emb_info}\n"
        f"  summary: {summary_preview!r}\n"
    )
    community_sample.append({
        "id": row["id"],
        "size": row["size"],
        "modularity": row["mod"],
        "summary_preview": summary_preview,
        "has_embedding": emb is not None,
    })

COMMUNITY nodes (top 5 by size):

  id=comm_01eff41340ab643…  size= 522  mod=0.4220
  emb: dims=1024 norm=1.0000
  summary: '根据所提供文献片段，核心主题为唐代均田制及相关土地制度问题，尤其聚焦于妇女受田、退田规则及户内田产分配。时代背景涉及北朝至隋唐均田令的演变，如隋炀帝废止妇人受田，唐代中女为户主时的给田与退田规定。学术意义在于通过敦煌吐鲁番文书与传世文献（如'

  id=comm_fd3e4c26bc4fa1e…  size= 298  mod=0.4220
  emb: dims=1024 norm=1.0000
  summary: '这些文献片段的核心主题是**唐代法典（特别是《永徽律》《垂拱律》及其疏议）的敦煌吐鲁番写本残卷**。时代背景为**唐代前期（高宗、武则天时期）**，主要涉及律、令、格、式等法律文本的抄写与流传。学术意义在于：这些出土文书（如P.3608、S'

  id=comm_b7b11695609b659…  size= 286  mod=0.4220
  emb: dims=1024 norm=1.0000
  summary: '这些文献片段的核心主题是**敦煌吐鲁番出土唐代法制文书**的研究，包括唐代法律条文、田令、户籍制度及社会经济文书。时代背景为**唐代（7—10世纪）**，涉及《唐研究》等学术著作及出土文献（如P.3690、72TAM230等编号）。学术意义'

  id=comm_20180ef55e69d1b…  size= 230  mod=0.4220
  emb: dims=1024 norm=1.0000
  summary: '这些文献片段虽存在文字识别错误或乱码，但核心主题指向**唐代敦煌吐鲁番文书**，涉及**经济制度、盐政运输、手工业、医疗记录、户籍管理**等内容。文献中反复出现的编号（如P.2507、P.2819、TAM等）是敦煌写本和吐鲁番阿斯塔那墓葬出'

  id=comm_97b4104bf8118bb…  size=   6  mod=0.4220
  emb: dims=1024 norm=1.0000
  summary: '这些文献片段的核心主题是**早期道教与

## 7. IN_COMMUNITY Edge Check


In [9]:
with driver.session() as s:
    edge_sample = s.run(
        "MATCH (ch:CHUNK)-[:IN_COMMUNITY]->(comm:COMMUNITY) "
        "RETURN ch.id AS chunk_id, comm.id AS community_id, comm.size AS comm_size "
        "ORDER BY comm.size DESC LIMIT 5"
    ).data()

    member_stats = s.run(
        "MATCH (ch:CHUNK)-[:IN_COMMUNITY]->(comm:COMMUNITY) "
        "RETURN count(ch) AS total_memberships, "
        "  count(DISTINCT ch) AS unique_chunks, "
        "  count(DISTINCT comm) AS unique_communities"
    ).single()

print("IN_COMMUNITY edge statistics:")
print(f"  Total memberships  : {member_stats['total_memberships']}")
print(f"  Unique chunks      : {member_stats['unique_chunks']}")
print(f"  Unique communities : {member_stats['unique_communities']}")
print()
print("Sample memberships (top communities by size):")
for row in edge_sample:
    print(
        f"  chunk={row['chunk_id'][:40]} "
        f"→ comm={row['community_id'][:20]}… (size={row['comm_size']})"
    )

IN_COMMUNITY edge statistics:
  Total memberships  : 1342
  Unique chunks      : 1342
  Unique communities : 5

Sample memberships (top communities by size):
  chunk=刘后滨_唐代选官政务研究_汇总稿2-1__a0a87ed317::p00054: → comm=comm_01eff41340ab643… (size=522)
  chunk=刘琴丽_唐代武官选任制度初探__4b2f2b3625::p00155::chun → comm=comm_01eff41340ab643… (size=522)
  chunk=刘后滨_唐代选官政务研究_汇总稿2-1__a0a87ed317::p00067: → comm=comm_01eff41340ab643… (size=522)
  chunk=北京大学中国中古史研究中心_敦煌吐鲁番文献研究论集第3辑__96128943a5 → comm=comm_01eff41340ab643… (size=522)
  chunk=刘琴丽_唐代武官选任制度初探__4b2f2b3625::p00255::chun → comm=comm_01eff41340ab643… (size=522)


## 8. Community Vector Search Probe

Embed a sample query and search `community_summary_embedding_index` — this is how the
search pipeline routes synthesis/interpretive queries through community-level context.


In [10]:
from apps.backend.pipeline.search import embed_query

_COMM_VECTOR_QUERY = """
CALL db.index.vector.queryNodes('community_summary_embedding_index', $top_k, $embedding)
YIELD node AS c, score
RETURN c.id AS id, c.size AS size, c.summary AS summary, score
"""

PROBE_QUERIES = [
    "唐代科舉制度與官員選拔的關係",
    "均田制土地分配的歷史演變",
    "安史之亂對唐代政治的影響",
]

probe_results: list[dict] = []

with driver.session() as s:
    comm_count = s.run("MATCH (c:COMMUNITY) WHERE c.embedding IS NOT NULL RETURN count(c) AS n").single()["n"]

if comm_count == 0:
    print("⚠️  No COMMUNITY nodes with embeddings yet — community search not available.")
    print("   This is expected if the smoke run wrote 0 communities (e.g. empty MENTION edges).")
else:
    print(f"Communities with embeddings: {comm_count}\n")
    for q in PROBE_QUERIES:
        vec = embed_query(q)
        with driver.session() as s:
            rows = s.run(_COMM_VECTOR_QUERY, top_k=3, embedding=vec).data()
        print(f"Query: {q!r}")
        for row in rows:
            summary_preview = (row["summary"] or "")[:100].replace("\n", " ")
            print(
                f"  score={row['score']:.4f}  size={row['size']:3d}  "
                f"summary={summary_preview!r}"
            )
        print()
        probe_results.append({"query": q, "hits": [
            {"id": r["id"], "size": r["size"], "score": r["score"],
             "summary_preview": (r["summary"] or "")[:80]}
            for r in rows
        ]})

Communities with embeddings: 5



2026-05-29 11:35:37,141 INFO     apps.backend.llm.silra: silra.embed model=text-embedding-v4 prompt=12 total=12


Query: '唐代科舉制度與官員選拔的關係'
  score=0.6709  size=286  summary='这些文献片段的核心主题是**敦煌吐鲁番出土唐代法制文书**的研究，包括唐代法律条文、田令、户籍制度及社会经济文书。时代背景为**唐代（7—10世纪）**，涉及《唐研究》等学术著作及出土文献（如P.36'
  score=0.6617  size=298  summary='这些文献片段的核心主题是**唐代法典（特别是《永徽律》《垂拱律》及其疏议）的敦煌吐鲁番写本残卷**。时代背景为**唐代前期（高宗、武则天时期）**，主要涉及律、令、格、式等法律文本的抄写与流传。学术意'
  score=0.6529  size=230  summary='这些文献片段虽存在文字识别错误或乱码，但核心主题指向**唐代敦煌吐鲁番文书**，涉及**经济制度、盐政运输、手工业、医疗记录、户籍管理**等内容。文献中反复出现的编号（如P.2507、P.2819、T'



2026-05-29 11:35:40,174 INFO     apps.backend.llm.silra: silra.embed model=text-embedding-v4 prompt=10 total=10


Query: '均田制土地分配的歷史演變'
  score=0.7611  size=522  summary='根据所提供文献片段，核心主题为唐代均田制及相关土地制度问题，尤其聚焦于妇女受田、退田规则及户内田产分配。时代背景涉及北朝至隋唐均田令的演变，如隋炀帝废止妇人受田，唐代中女为户主时的给田与退田规定。学术'
  score=0.6878  size=286  summary='这些文献片段的核心主题是**敦煌吐鲁番出土唐代法制文书**的研究，包括唐代法律条文、田令、户籍制度及社会经济文书。时代背景为**唐代（7—10世纪）**，涉及《唐研究》等学术著作及出土文献（如P.36'
  score=0.6576  size=298  summary='这些文献片段的核心主题是**唐代法典（特别是《永徽律》《垂拱律》及其疏议）的敦煌吐鲁番写本残卷**。时代背景为**唐代前期（高宗、武则天时期）**，主要涉及律、令、格、式等法律文本的抄写与流传。学术意'



2026-05-29 11:35:42,143 INFO     apps.backend.llm.silra: silra.embed model=text-embedding-v4 prompt=10 total=10


Query: '安史之亂對唐代政治的影響'
  score=0.6620  size=298  summary='这些文献片段的核心主题是**唐代法典（特别是《永徽律》《垂拱律》及其疏议）的敦煌吐鲁番写本残卷**。时代背景为**唐代前期（高宗、武则天时期）**，主要涉及律、令、格、式等法律文本的抄写与流传。学术意'
  score=0.6562  size=286  summary='这些文献片段的核心主题是**敦煌吐鲁番出土唐代法制文书**的研究，包括唐代法律条文、田令、户籍制度及社会经济文书。时代背景为**唐代（7—10世纪）**，涉及《唐研究》等学术著作及出土文献（如P.36'
  score=0.6445  size=230  summary='这些文献片段虽存在文字识别错误或乱码，但核心主题指向**唐代敦煌吐鲁番文书**，涉及**经济制度、盐政运输、手工业、医疗记录、户籍管理**等内容。文献中反复出现的编号（如P.2507、P.2819、T'



## 9. Artefact Write


In [11]:
artifact = {
    "phase": "08c_community_summaries",
    "ts": datetime.now(timezone.utc).isoformat(),
    "preflight": preflight,
    "graph_load": {
        "smoke_limit": SMOKE_LIMIT,
        "n_chunks": n_chunks,
        "n_keywords": n_keywords,
        "n_edges": len(edge_list),
    },
    "leiden": {
        "resolution": _LEIDEN_RESOLUTION,
        "min_size": _MIN_COMMUNITY_SIZE,
        "communities_detected": len(communities),
        "modularity": round(modularity, 6),
    },
    "smoke_run": report.to_dict(),
    "community_sample": community_sample,
    "probe_results": probe_results,
    "final_counts": {
        "community_nodes": n_comm_after,
        "in_community_edges": n_member_after,
    },
}

artifact_path = ARTIFACT_DIR / "community_summaries.json"
artifact_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f"Artifact written → {artifact_path}")
print(f"File size: {artifact_path.stat().st_size:,} bytes")

print()
print("=" * 60)
print("Community summaries (B3) status")
print(f"  Communities written  : {report.communities_written}")
print(f"  Memberships written  : {report.memberships_written}")
print(f"  Errors               : {len(report.errors)}")
if report.errors:
    for e in report.errors[:3]:
        print(f"    {e}")
print()
print("  Full corpus run (all 270K MENTION edges):")
print("    caffeinate -dimsu uv run python scripts/run_communities.py \\ ")
print("      --log-file logs/community_summaries.log")
print("=" * 60)

Artifact written → /Users/mohasani/Ancient/notebooks/_artifacts/08c_community_summaries/community_summaries.json
File size: 7,017 bytes

Community summaries (B3) status
  Communities written  : 5
  Memberships written  : 1342
  Errors               : 0

  Full corpus run (all 270K MENTION edges):
    caffeinate -dimsu uv run python scripts/run_communities.py \ 
      --log-file logs/community_summaries.log
